# 5.3 1D CNN Time Series

這份 Notebook 使用 Conv1D 做時間序列分類，示範如何擷取局部 pattern。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

tf.keras.utils.set_random_seed(42)


## 2. 建立局部 pattern 資料

三類序列分別包含 bump、dip 與 double_peak。這類資料適合用 1D CNN 擷取局部形狀。


In [ ]:
CLASS_NAMES = np.array(['bump', 'dip', 'double_peak'])
TIMESTEPS = 64

def make_pattern(label, rng, timesteps=TIMESTEPS):
    x_axis = np.arange(timesteps)
    base = rng.normal(0, 0.08, timesteps)
    if label == 0:
        center = rng.integers(22, 42)
        value = base + 1.5 * np.exp(-0.5 * ((x_axis - center) / 4.0) ** 2)
    elif label == 1:
        center = rng.integers(22, 42)
        value = base - 1.5 * np.exp(-0.5 * ((x_axis - center) / 4.0) ** 2)
    else:
        c1 = rng.integers(14, 26)
        c2 = rng.integers(38, 50)
        value = base + 1.2 * np.exp(-0.5 * ((x_axis - c1) / 3.0) ** 2) + 1.2 * np.exp(-0.5 * ((x_axis - c2) / 3.0) ** 2)
    return value.astype('float32')[..., np.newaxis]

def make_dataset(samples_per_class=260, seed=42):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            X.append(make_pattern(label, rng))
            y.append(label)
    X = np.stack(X).astype('float32')
    y = np.array(y, dtype='int64')
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

X, y = make_dataset(samples_per_class=260, seed=21)
n = len(X)
train_end = int(n * 0.7)
valid_end = int(n * 0.85)
x_train, y_train = X[:train_end], y[:train_end]
x_valid, y_valid = X[train_end:valid_end], y[train_end:valid_end]
x_test, y_test = X[valid_end:], y[valid_end:]
mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
x_train = (x_train - mean) / std
x_valid = (x_valid - mean) / std
x_test = (x_test - mean) / std
print(x_train.shape, x_valid.shape, x_test.shape)


## 3. 觀察局部 pattern


In [ ]:
plt.figure(figsize=(9, 3))
for label in range(len(CLASS_NAMES)):
    idx = np.where(y == label)[0][0]
    plt.plot(X[idx, :, 0], label=CLASS_NAMES[label])
plt.title('Local patterns')
plt.legend()
plt.show()


## 4. 建立 1D CNN 模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=x_train.shape[1:]),
    tf.keras.layers.Conv1D(32, 5, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 5. 訓練與評估


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=25,
    batch_size=32,
    verbose=0,
)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Conv1D accuracy')
plt.ylim(0, 1.05)
plt.show()

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(model.evaluate(x_test, y_test, verbose=0, return_dict=True))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 6. 如何套用自己的資料

若任務由局部形狀決定，例如尖峰、凹陷、短暫震盪或固定波形，可以先嘗試 1D CNN。多感測器資料可把每個感測器放在 features 維度。


## 7. 小結

1D CNN 會沿時間軸擷取局部 pattern，適合訓練快速、局部特徵清楚的時間序列分類任務。
